In [23]:
import torch
from torch import nn
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import numpy as np
import regex as re
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

In [5]:
df = pd.read_csv("../data/aes_dataset_5k_clean.csv")
df = df[df['dataset'] == 'analisis_essay'][['reference_answer', 'answer', 'score', 'normalized_score', 'dataset', 'dataset_num']]
print(df.info())
df.head()

<class 'pandas.core.frame.DataFrame'>
Index: 2162 entries, 0 to 2161
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   reference_answer  2162 non-null   object 
 1   answer            2162 non-null   object 
 2   score             2162 non-null   float64
 3   normalized_score  2162 non-null   float64
 4   dataset           2162 non-null   object 
 5   dataset_num       2162 non-null   object 
dtypes: float64(2), object(4)
memory usage: 118.2+ KB
None


,reference_answer,answer,score,normalized_score,dataset,dataset_num
0,Fungsi karbohidrat adalah sebagai pemasok ener...,"sumber tenaga, pemanis alami, menjaga sistem i...",27.0,0.27,analisis_essay,analisis_essay-1
1,Fungsi karbohidrat adalah sebagai pemasok ener...,"sebagai sumber energi, pemanis alami, menjaga ...",21.0,0.21,analisis_essay,analisis_essay-1
2,Fungsi karbohidrat adalah sebagai pemasok ener...,1. Sebagai energi. 2. Sebagai memperlancaar pe...,42.0,0.42,analisis_essay,analisis_essay-1
3,Fungsi karbohidrat adalah sebagai pemasok ener...,"untuk membuat kenyang, agar tidak lapar, agar ...",18.0,0.18,analisis_essay,analisis_essay-1
4,Fungsi karbohidrat adalah sebagai pemasok ener...,Karbohidrat mempunyai peran penting untuk pros...,82.0,0.82,analisis_essay,analisis_essay-1


In [12]:
def split_dataset(df, train_ratio, valid_ratio, test_ratio):
    print("run split dataset...")
    subset_dataset = df['dataset_num'].unique()
    splits = {}
    for subset in subset_dataset:
        # get data by dataset_num
        subset_df = df[df['dataset_num'] == subset]

        # split dataset
        train_df, temp_df = train_test_split(subset_df, test_size=(1 - train_ratio), random_state=SEED, shuffle=True)
        valid_df, test_df = train_test_split(temp_df, test_size=test_ratio / (valid_ratio + test_ratio), random_state=SEED, shuffle=True)

        # save split dataset
        splits[subset] = {
            'train': train_df,
            'valid': valid_df,
            'test': test_df,
        }
    
    train_dataset = pd.concat([splits[subset]['train'] for subset in subset_dataset])
    valid_dataset = pd.concat([splits[subset]['valid'] for subset in subset_dataset])
    test_dataset = pd.concat([splits[subset]['test'] for subset in subset_dataset])

    return train_dataset, valid_dataset, test_dataset

train_dataset, valid_dataset, test_dataset = split_dataset(df, 0.8, 0.1, 0.1)
print("train dataset : ", train_dataset.shape)
print("valid dataset : ", valid_dataset.shape)
print("test dataset : ", test_dataset.shape)

run split dataset...
train dataset :  (1711, 6)
valid dataset :  (213, 6)
test dataset :  (238, 6)


## Dataset

In [14]:
class AnswerScoringDataset(Dataset):
    """
    Dataset class for answer scoring using your specific DataFrame structure.
    """
    def __init__(self, dataframe, use_normalized_score=True):
        self.reference_answers = dataframe['reference_answer'].tolist()
        self.student_answers = dataframe['answer'].tolist()  # Note the column name change
        
        # Use either normalized or raw score based on parameter
        if use_normalized_score:
            self.scores = dataframe['normalized_score'].values.astype(np.float32)
        else:
            self.scores = dataframe['score'].values.astype(np.float32)
    
    def preprocess_text(self, text):
        # Remove extra whitespace
        text = ' '.join(text.split())
        # Convert to lowercase
        text = text.lower()
        # Remove special characters (keep punctuation)
        text = re.sub(r'[^a-zA-Z0-9\s.,!?]', '', text)
        return text
    
    def __len__(self):
        return len(self.scores)
    
    def __getitem__(self, idx):
        return {
            'reference_answer': self.preprocess_text(self.reference_answers[idx]),
            'student_answer': self.preprocess_text(self.student_answers[idx]),
            'score': torch.tensor(self.scores[idx], dtype=torch.float)
        }
    
train_dataset = AnswerScoringDataset(train_dataset)
val_dataset = AnswerScoringDataset(valid_dataset)
test_dataset = AnswerScoringDataset(test_dataset)

In [17]:
class SiameseScoringModel(nn.Module):
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        super(SiameseScoringModel, self).__init__()
        # Load the sentence transformer model
        self.sentence_transformer = SentenceTransformer(model_name)
        self.embedding_dim = self.sentence_transformer.get_sentence_embedding_dimension()
        
        # dropout layer
        self.dropout = nn.Dropout(p=0.1, inplace=False)
        # Regression head
        self.regression_head = nn.Linear(self.embedding_dim * 2, 1)

        
    def forward(self, reference_texts, student_texts):
        reference_features = self.sentence_transformer.tokenize(reference_texts)
        student_features = self.sentence_transformer.tokenize(student_texts)

        if next(self.parameters()).device != reference_features['input_ids'].device:
            reference_features = {k: v.to(next(self.parameters()).device) for k, v in reference_features.items()}
            student_features = {k: v.to(next(self.parameters()).device) for k, v in student_features.items()}
        
        # Get embeddings while maintaining the computation graph
        reference_embeddings = self.sentence_transformer.forward(reference_features)['sentence_embedding']
        student_embeddings = self.sentence_transformer.forward(student_features)['sentence_embedding']
        
        # Concatenate the embeddings
        combined = torch.cat((reference_embeddings, student_embeddings), dim=1)
        
        x = self.dropout(combined)
        # Final regression score
        score = torch.sigmoid(self.regression_head(x))
        
        return score
    
model = SiameseScoringModel().to(device)

In [16]:
def collate_fn(batch):
    reference_answers = [item['reference_answer'] for item in batch]
    student_answers = [item['student_answer'] for item in batch]
    scores = torch.tensor([item['score'] for item in batch], dtype=torch.float).unsqueeze(1)
    
    return {
        'reference_answers': reference_answers,
        'student_answers': student_answers,
        'scores': scores
    }

# Create data loaders
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=batch_size, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=batch_size, collate_fn=collate_fn)

## Pipeline

In [18]:
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
epochs = 1

In [28]:
for epoch in range(epochs):
    model.train()
    train_mse_loss = 0.0
    all_predictions = []
    all_targets = []

    for batch in train_loader:
        reference_answers = batch['reference_answers']
        student_answers = batch['student_answers']
        scores = batch['scores'].to(device)
        
        # Zero the parameter gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(reference_answers, student_answers)
        loss = criterion(outputs, scores)

        # Backward pass and optimize
        loss.backward()
        # Gradient clipping to prevent explosive gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        # save training loss and metrik evaluation
        train_mse_loss += loss.item()
        all_predictions.extend(outputs.detach().cpu().numpy())
        all_targets.extend(batch['scores'].detach().cpu().numpy())

    avg_train_loss = train_mse_loss / len(train_loader)
    mae = mean_absolute_error(all_targets, all_predictions)
    rmse = np.sqrt(mean_squared_error(all_targets, all_predictions))
    # change array dim from 1 to 0
    targets_flat = [t.item() for t in all_targets]
    predictions_flat = [p.item() for p in all_predictions]
    pearson_corr, _ = pearsonr(targets_flat, predictions_flat)
    print(f"Epoch {epoch+1}/{epochs} - Avg training loss: {avg_train_loss:.4f}, MAE: {mae:.4}, RMSE: {rmse:.4}, Pearson Corr: {pearson_corr:.4}")

    # =============== EVAL PROCESS
    model.eval()
    total_mse_loss = 0
    all_predictions = []
    all_targets = []
    with torch.no_grad():
        for batch in val_loader:
            # move to device
            reference_answers = batch['reference_answers']
            student_answers = batch['student_answers']
            scores = batch['scores'].to(device)

            # get prediction
            # Forward pass
            outputs = model(reference_answers, student_answers)
            loss = criterion(outputs, scores)

            # save training loss and metrik evaluation
            total_mse_loss += loss.item()
            all_predictions.extend(outputs.detach().cpu().numpy())
            all_targets.extend(batch['scores'].detach().cpu().numpy())

        avg_mse_loss = total_mse_loss / len(val_loader)
        mae = mean_absolute_error(all_targets, all_predictions)
        rmse = np.sqrt(mean_squared_error(all_targets, all_predictions))
        # change array dim from 1 to 0
        targets_flat = [t.item() for t in all_targets]
        predictions_flat = [p.item() for p in all_predictions]
        pearson_corr, _ = pearsonr(targets_flat, predictions_flat)
        print(f"Avg validation loss: {avg_mse_loss:.4f}, MAE: {mae:.4}, RMSE: {rmse:.4}, Pearson Corr: {pearson_corr:.4}")
        

Epoch 1/1 - Avg training loss: 0.0307, MAE: 0.1407, RMSE: 0.1751, Pearson Corr: 0.8393
Avg validation loss: 0.0359, MAE: 0.1541, RMSE: 0.1905, Pearson Corr: 0.7252
